# VLA Workshop — robosuite × SmolVLA

Google Colabで、環境確認からVLA推論まで順に取り組みます。処理本体は共通のPythonパッケージです。

1. ランタイム → ランタイムのタイプを変更 → GPUを選びます（T4 / L4など）。
2. このノートブックを自分のDriveにコピーし、上から順番に実行します。
3. 最初は00と01を各20ステップだけ試します。タスク成功は今回の合格条件ではありません。
4. 最後に動画・ログをダウンロードし、ランタイムを接続解除して削除します。

Python 3.12向けです。初回セットアップとモデル取得には数分以上かかる場合があります。

In [ ]:
# GPUとPythonを確認する
import subprocess
import sys

print(sys.version)
subprocess.run(["nvidia-smi"], check=True)
assert sys.version_info[:2] == (3, 12), "Python 3.12のランタイムを使ってください"

## 教材コードを取得する

GitHub配布時は講師が下の`REPOSITORY`・`COMMIT`・`PROJECT_SUBDIR`を設定します。
教材のcommitを固定し、参加者は同じ版の共通コードを使います。

配布先が未設定の場合は、講師が配布した`vla_workshop.zip`をアップロードしてください。
GitHubがprivateの場合も、認証情報を書き込まずZIP配布で始められます。

In [ ]:
from pathlib import Path
import hashlib
import io
import json
import os
import zipfile

# 講師設定：GitHub配布先が決まったら、この3項目を設定します。
REPOSITORY = "takumiecd/vla-workshop"             # 例: team/robot-learning-workshop（公開リポジトリ）
COMMIT = "1e2924947464dacece85c2b52a9f66669d39eb4d"                 # 教材を配布するcommitの40文字SHA
PROJECT_SUBDIR = "vla_workshop"  # リポジトリ直下に置く場合は "."

prepared = os.environ.get("VLA_WORKSHOP_PROJECT")
if prepared:
    PROJECT = Path(prepared).resolve()
elif REPOSITORY:
    assert len(COMMIT) == 40 and all(c in "0123456789abcdef" for c in COMMIT), "教材commitを固定してください"
    checkout = Path("/content") / f"workshop-{COMMIT[:12]}"
    if not checkout.exists():
        subprocess.run(["git", "init", str(checkout)], check=True)
        subprocess.run(["git", "-C", str(checkout), "remote", "add", "origin",
                        f"https://github.com/{REPOSITORY}.git"], check=True)
        subprocess.run(["git", "-C", str(checkout), "fetch", "--depth", "1", "origin", COMMIT], check=True)
        subprocess.run(["git", "-C", str(checkout), "checkout", "--detach", "FETCH_HEAD"], check=True)
    actual = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
    assert actual == COMMIT, "取得した教材のcommitが一致しません"
    PROJECT = checkout / PROJECT_SUBDIR
else:
    PROJECT = Path("/content/vla_workshop")
    if not (PROJECT / "bundle_manifest.json").exists():
        from google.colab import files
        uploaded = files.upload()
        assert len(uploaded) == 1, "vla_workshop.zipを1つ選んでください"
        with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
            for name in archive.namelist():
                target = (PROJECT.parent / name).resolve()
                assert target.is_relative_to(PROJECT), f"教材外のパスです: {name}"
            archive.extractall(PROJECT.parent)
    manifest = json.loads((PROJECT / "bundle_manifest.json").read_text())
    for name, digest in manifest.items():
        assert hashlib.sha256((PROJECT / name).read_bytes()).hexdigest() == digest, name

assert (PROJECT / "pyproject.toml").is_file(), "教材フォルダを確認してください"
print("教材:", PROJECT)

## 共通環境をセットアップする

セットアップ用スクリプトが、この教材専用のPython環境を用意します。
Colab標準のCUDA/PyTorchを利用し、追加ライブラリを教材用の環境に入れます。実習は毎回そのPythonから実行します。
途中で失敗した場合は、表示される`outputs/setup.log`の末尾を確認してください。

In [ ]:
subprocess.run([sys.executable, str(PROJECT / "scripts/setup_colab.py"),
                "--project", str(PROJECT)], check=True)
PYTHON = PROJECT / ".venv/bin/python"
print("実習用Python:", PYTHON)

## 00: 環境と観測を確認する

標準Lift環境にPandaを置きます。画像・状態・制御・動画保存を確認します。
この段階は学習モデルを使わず、手先の移動指令をゼロ、グリッパーを開く指令にします。

In [ ]:
def run_lesson(script, *, steps=20, episodes=1):
    result = subprocess.run(
        [str(PYTHON), str(PROJECT / script), "--config", str(PROJECT / "configs/lift.json"),
         "--max-steps", str(steps), "--episodes", str(episodes)],
        cwd=PROJECT, text=True, capture_output=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr[-6000:])
    result.check_returncode()
    line = next(line for line in result.stdout.splitlines() if line.startswith("Results: "))
    return Path(line.removeprefix("Results: "))

inspect_result = run_lesson("lessons/00_environment/run.py")

In [ ]:
from IPython.display import Image, Video, display

display(Image(filename=str(inspect_result / "episode_000/agentview.png")))
display(Video(str(inspect_result / "episode_000/video.mp4"), embed=True))
print((inspect_result / "summary.json").read_text())
print((inspect_result / "episode_000/environment.json").read_text())

### 確認課題

- 画像にはPandaと立方体が写っていますか。
- 8次元の状態と7次元の行動について、各要素の意味を説明できますか。
- 20 Hzという設定と、このセルの実行時間の違いを説明できますか。

詳しい仕様は教材内の`docs/io_contract.md`、読むコードは`src/vla_workshop/envs/robosuite_env.py`です。

## 01: CUDAでSmolVLAを推論する

LIBERO用チェックポイントの画像2枚・8次元状態・7次元行動に合わせて接続します。
モデル付属の正規化・逆正規化を使い、実行後は再び観測して次の行動を選びます。

この実習のシーンはLiftです。ここでの成否をLIBEROの成功率として扱わないでください。

In [ ]:
MAX_STEPS = 20
EPISODES = 1

inference_result = run_lesson("lessons/01_smolvla_inference/run.py",
                              steps=MAX_STEPS, episodes=EPISODES)

In [ ]:
display(Video(str(inference_result / "episode_000/video.mp4"), embed=True))
print((inference_result / "summary.json").read_text())
run_info = json.loads((inference_result / "run.json").read_text())
print(json.dumps(run_info["policy"], indent=2, ensure_ascii=False))
steps = [json.loads(line) for line in
         (inference_result / "episode_000/steps.jsonl").read_text().splitlines()]
print("最初のstep:", json.dumps(steps[0], indent=2, ensure_ascii=False))

### 確認課題

- 実際のデバイスが`cuda:0`になっていますか。
- モデルの行動と環境に渡した行動は一致していますか。clipされた場合はどの要素ですか。
- 失敗した場合、動画から何が起きたか説明できますか。
- 20ステップの確認後、`MAX_STEPS=200`に変えて試してください。`EPISODES=3`ではseedも変わります。

今回の合格条件は「観測→推論→行動実行を通し、動画とログを保存し、内容を説明できること」です。

## 結果を保存して終了する

Colabのローカルファイルはランタイム削除時に失われます。動画・ログ・実行設定をZIPでダウンロードしてください。
ダウンロード後、ランタイムの「接続解除して削除」を実行します。

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive("/content/vla_workshop_results", "zip", PROJECT / "outputs")
files.download(archive)